# NOOTEBOOK DE CONFECCION DE SOLAPAMINETO DE BANDAS

## Aqu
ASD}

In [ ]:
import re
from pathlib import Path
from collections import defaultdict
import rasterio
from rasterio.merge import merge

# Ruta donde están tus imágenes
BASE_DIR = Path(r"E:\Silos\Base de datos\procesado_sin_nubes")
OUT_DIR = BASE_DIR / "mosaics"
OUT_DIR.mkdir(exist_ok=True)

# Extensiones a considerar (añade las que necesites)
EXTS = (".tif", ".tiff", ".jp2", ".TIF", ".JP2")

# Expresión regular para extraer el identificador de escena (ej: 2020_6_18_2)
# Ajusta si tus nombres cambian. Esta busca 4 dígitos (año) seguido de 1+ dígitos para mes/día/índice.
pattern = re.compile(r".*?(\d{4}_\d{1,2}_\d{1,2}_\d+).*")

# Recolectar archivos y agrupar por escena
groups = defaultdict(list)
for p in BASE_DIR.iterdir():
    if p.is_file() and p.suffix in EXTS:
        m = pattern.search(p.name)
        if m:
            key = m.group(1)   # p.ej. "2020_6_18_2"
            groups[key].append(p)
        else:
            print(f"⚠️  Archivo sin patrón reconocido (se saltará): {p.name}")

print(f"Grupos detectados: {len(groups)}")

# Función para mosaicar una lista de archivos y salvar un GeoTIFF
def mosaic_and_save(file_list, out_path):
    srcs = []
    try:
        for fp in file_list:
            src = rasterio.open(fp)
            srcs.append(src)

        # comprobar que todos tienen el mismo CRS y número de bandas
        crs0 = srcs[0].crs
        bands0 = srcs[0].count
        for s in srcs[1:]:
            if s.crs != crs0:
                print(f"❗ CRS distinto en {s.name} ({s.crs}); intenta reproyectar o revisar.")
            if s.count != bands0:
                print(f"❗ Diferente número de bandas en {s.name} ({s.count} vs {bands0}).")

        mosaic_arr, out_trans = merge(srcs)  # devuelve array (bands, height, width)
        out_meta = srcs[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic_arr.shape[1],
            "width": mosaic_arr.shape[2],
            "transform": out_trans,
            "count": mosaic_arr.shape[0]
        })

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(mosaic_arr)

        print(f"✅ Mosaico guardado: {out_path.name}  (bands={mosaic_arr.shape[0]}, {mosaic_arr.shape[1]}x{mosaic_arr.shape[2]})")

    except Exception as e:
        print(f"❌ Error mosaicing {file_list}: {e}")
    finally:
        for s in srcs:
            try:
                s.close()
            except:
                pass

# Ejecutar para cada grupo que tenga al menos 2 archivos (o 3 como tu caso)
for key, files in groups.items():
    if len(files) < 1:
        continue
    # ordenar para reproducibilidad
    files = sorted(files)
    out_fp = OUT_DIR / f"{key}_mosaic.tif"
    # Si ya existe, saltar (quitar esta condición si quieres sobreescribir)
    if out_fp.exists():
        print(f"⏭️  Ya existe {out_fp.name}, saltando.")
        continue
    mosaic_and_save(files, out_fp)

print("Proceso terminado.")


Grupos detectados: 328
✅ Mosaico guardado: 2020_6_10_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_10_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_13_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_13_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_15_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_15_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_18_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_18_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_18_2_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_20_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_20_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_23_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_23_1_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_25_0_mosaic.tif  (bands=4, 30978x10980)
✅ Mosaico guardado: 2020_6_25_1_mosaic.